In [1]:
from collections import deque
import numpy as np
import cv2

In [2]:
buffer_size = 64
TARGET_COLOR = "blue"  # green, blue, red.
# Multiple colors need to be defined incase ball is in different color.
color_ranges = {
    "green": ((29, 86, 6), (64, 255, 255)),
    "blue":  ((90, 80, 50), (130, 255, 255)),
    }
red_lower1 = (0, 120, 70)
red_upper1 = (10, 255, 255)
red_lower2 = (170, 120, 70)
red_upper2 = (180, 255, 255)

# Tracked points.
pts = deque(maxlen=buffer_size)
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (600, 400))

    blurred = cv2.GaussianBlur(frame, (11, 11), 0)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

    # Masking
    if TARGET_COLOR == "red":
        mask1 = cv2.inRange(hsv, red_lower1, red_upper1)
        mask2 = cv2.inRange(hsv, red_lower2, red_upper2)
        mask = cv2.bitwise_or(mask1, mask2)
    else:
        lower, upper = color_ranges[TARGET_COLOR]
        mask = cv2.inRange(hsv, lower, upper)
    
    # Noise Cleaning
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    
    center = None
    # Proceed if one contour is found.
    if len(cnts) > 0:
     c = max(cnts, key=cv2.contourArea)
     ((x,y), radius) = cv2.minEnclosingCircle(c)
     M = cv2.moments(c)
     center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))
     # Proceed if radius meets minimum size
     if radius > 10:
         # draw circle and centroid around frame
         cv2.circle(frame, (int(x), int(y)), int(radius)
                    , (0,255,255),2)
         cv2.circle(frame,center,5,(0,0,255),-1)

    cv2.imshow("Frame",frame)
    cv2.imshow("Mask",mask)
    
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cap.release()
cv2.destroyAllWindows()
    